# Milestone 2 Objective

Build and validate the Repository Analyzer Agent as a standalone agent before integrating it into the full workflow.

## Repository Analyzer Agent Role

The agent inspects repository files and returns compact metadata: language, framework, candidate route files, candidate UI files, test evidence, documentation candidates, endpoints, UI flows, and risks.

## Why It Works Alone Before Integration

Each agent must be testable by itself first. The analyzer produces JSON evidence without relying on RAG, MCP, Selenium, Locust, dashboards, or real LLM calls.

## Mini LangGraph Workflow

```text
START -> repo_analyzer -> END
```

## What The Agent Writes To State

The Repository Analyzer writes `repo_path`, `project_info`, `discovered_endpoints`, `discovered_ui_flows`, and `indexed_documents` to State. These are compact summaries and paths, not large file contents.

## Security Note

The analyzer reads text files but does not execute the target project, does not call LLM APIs, and does not print secrets.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
project_root = cwd if (cwd / "src").exists() else cwd.parent
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

from test_auto.agents.repo_analyzer import run_repo_analyzer_alone

In [2]:
from tempfile import TemporaryDirectory

tmp = TemporaryDirectory()
repo = Path(tmp.name) / "fake_todo_repo"

def write_file(relative_path: str, content: str = "") -> None:
    path = repo / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")

write_file("README.md", "# Fake Todo API\n")
write_file(
    "requirements.txt",
    "django\ndjangorestframework\ndjangorestframework-simplejwt\npytest\n",
)
write_file("manage.py", "# manage.py placeholder\n")
write_file(
    "todo/urls.py",
    '\n'.join([
        "from django.urls import path",
        "urlpatterns = [",
        '    path("api/todos/", views.todo_list),',
        '    path("api/todos/<int:pk>/", views.todo_detail),',
        "]",
    ]),
)
write_file("todo/views.py", "def todo_list(request): pass\n")
write_file("templates/login.html", "<form>login</form>\n")
write_file("tests/test_todo.py", "def test_todo(): assert True\n")

repo

WindowsPath('C:/Users/malak/AppData/Local/Temp/tmpu9liimpp/fake_todo_repo')

In [3]:
repo_analyzer_result = run_repo_analyzer_alone(repo_path=str(repo))

In [4]:
import json

print(json.dumps(repo_analyzer_result, indent=2))

{
  "run_id": "run_20260520T083954Z_47ce16e1",
  "repo_path": "C:\\Users\\malak\\AppData\\Local\\Temp\\tmpu9liimpp\\fake_todo_repo",
  "project_info": {
    "language": "Python",
    "framework": "Django REST Framework",
    "test_framework": "pytest",
    "has_api": true,
    "has_ui": true,
    "auth_type": "JWT",
    "package_manager": "pip",
    "source_dirs": [
      "todo"
    ],
    "test_dirs": [
      "tests",
      "tests/test_todo.py"
    ],
    "candidate_docs": [
      "README.md"
    ],
    "candidate_api_files": [
      "todo/urls.py",
      "todo/views.py"
    ],
    "candidate_ui_files": [
      "templates/login.html"
    ],
    "risks": []
  },
  "discovered_endpoints": [
    {
      "name": "api_todos",
      "method": "UNKNOWN",
      "path": "/api/todos/",
      "source_file": "todo/urls.py",
      "line_number": 3,
      "confidence": 0.7
    },
    {
      "name": "api_todos_int:pk",
      "method": "UNKNOWN",
      "path": "/api/todos/<int:pk>/",
      "source_f

In [5]:
framework = repo_analyzer_result["project_info"]["framework"]
assert framework in {"Django REST Framework", "Django"}
framework

'Django REST Framework'

In [6]:
repo_analyzer_result["discovered_endpoints"]

[{'name': 'api_todos',
  'method': 'UNKNOWN',
  'path': '/api/todos/',
  'source_file': 'todo/urls.py',
  'line_number': 3,
  'confidence': 0.7},
 {'name': 'api_todos_int:pk',
  'method': 'UNKNOWN',
  'path': '/api/todos/<int:pk>/',
  'source_file': 'todo/urls.py',
  'line_number': 4,
  'confidence': 0.7}]

In [7]:
repo_analyzer_result["discovered_ui_flows"]

[{'name': 'login',
  'source_file': 'templates/login.html',
  'flow_type': 'authentication',
  'confidence': 0.6}]

In [8]:
from test_auto.graph.repo_analyzer_workflow import run_repo_analyzer_workflow

In [9]:
initial_state = {
    "run_id": "notebook_repo_analyzer_run",
    "repo_path": str(repo),
    "errors": [],
    "agent_logs": [],
}

final_state = run_repo_analyzer_workflow(initial_state)

In [10]:
print(json.dumps(final_state, indent=2))

{
  "run_id": "notebook_repo_analyzer_run",
  "errors": [],
  "agent_logs": [
    {
      "agent": "repo_analyzer",
      "timestamp": "2026-05-20T08:39:54.896344+00:00",
      "status": "success",
      "duration_seconds": 0.004685199994128197,
      "project_info": {
        "language": "Python",
        "framework": "Django REST Framework",
        "test_framework": "pytest",
        "has_api": true,
        "has_ui": true,
        "auth_type": "JWT",
        "package_manager": "pip",
        "source_dirs": [
          "todo"
        ],
        "test_dirs": [
          "tests",
          "tests/test_todo.py"
        ],
        "candidate_docs": [
          "README.md"
        ],
        "candidate_api_files": [
          "todo/urls.py",
          "todo/views.py"
        ],
        "candidate_ui_files": [
          "templates/login.html"
        ],
        "risks": []
      },
      "discovered_endpoints": [
        {
          "name": "api_todos",
          "method": "UNKNOWN",
    

In [11]:
{
    "project_info_json": repo_analyzer_result["project_info_path"],
    "repo_analyzer_result_json": repo_analyzer_result["agent_output_path"],
}

{'project_info_json': 'results\\runs\\run_20260520T083954Z_47ce16e1\\project_info.json',
 'repo_analyzer_result_json': 'results\\runs\\run_20260520T083954Z_47ce16e1\\repo_analyzer_result.json'}